# APX (Approximate Next-Token Probability) — LLaMA-2

Computes APX scores for LLaMA-2 as a faster alternative to full PLL.
APX uses next-token prediction probability (perplexity-based) rather than masked scoring.

APX(s) = PPL(sentence) = exp(-1/n * sum log P(w_i | w_<i))

In [ ]:
!pip install peft accelerate bitsandbytes transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch
import math
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!huggingface-cli login

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/NewStreoSet_Dataset.xlsx')
print(df.shape)

In [ ]:
MODEL_NAME = 'meta-llama/Llama-2-7b-hf'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
model.eval()

In [ ]:
def compute_ppl(sentence, tokenizer, model, device='cuda'):
 """Compute sentence perplexity (APX approximation)."""
 inputs = tokenizer(sentence, return_tensors='pt').to(device)
 with torch.no_grad():
 loss = model(**inputs, labels=inputs['input_ids']).loss
 return math.exp(loss.item())

def compute_apx(sentence, tokenizer, model, mean_target_ppl, device='cuda'):
 """APX = PPL(sentence) / MeanPPL(target across contexts)"""
 ppl = compute_ppl(sentence, tokenizer, model, device)
 return ppl / mean_target_ppl * 100 if mean_target_ppl > 0 else float('nan')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ppl_scores = []
for _, row in tqdm(df.iterrows(), total=len(df)):
 ppl_scores.append(compute_ppl(row['full_sentence'], tokenizer, model, device))
df['PPL_English'] = ppl_scores

In [ ]:
# Compute MeanTargetPPL per target word group
mean_ppl = df.groupby('target')['PPL_English'].mean().to_dict()
df['MeanTargetPPL_English'] = df['target'].map(mean_ppl)
df['APX_English'] = df['PPL_English'] / df['MeanTargetPPL_English'] * 100
df[['target','label_name','bias_type','PPL_English','MeanTargetPPL_English','APX_English']].head(10)

In [ ]:
# APX bias analysis by bias type
print('--- APX Bias by Type (LLaMA-2) ---')
for btype, group in df.groupby('bias_type'):
 stereo = group[group['label_name']=='stereotype']['APX_English'].mean()
 anti = group[group['label_name']=='anti-stereotype']['APX_English'].mean()
 print(f'{btype}: stereo={stereo:.2f}, anti={anti:.2f}, diff={stereo-anti:.2f}')

In [ ]:
df.to_csv('/content/drive/MyDrive/APX_LLaMA.csv')
print('Saved APX_LLaMA.csv')